In [2]:
import math
import pandas as pd 
import sys
import os

In [3]:
import sys
sys.path.append('../')

from src.PreProcessingPipeline import BM25_PreProcess

In [4]:
base_dir = os.getcwd()
csv_path = os.path.join(base_dir, "..", "data", "archive-5", "archive (2)", "bbc-news-data.csv")
df= pd.read_csv(csv_path, sep="\t")

In [5]:
# df = pd.read_csv("../data/archive(5)/archive (2)/bbc-news-data.csv",sep="\t")

df['document'] = df['title'] + " " + df ['content']

corpus = df['document'].to_list()

In [6]:
bm_prepro = BM25_PreProcess(corpus=corpus, set_stemming=True, set_lemmatization=False, set_stopwords=True)
processed_corpus = bm_prepro.get_corpus()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/arturaliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
len(corpus)

2225

In [8]:
class UnigramLM:
    def __init__(self, corpus, mu):
        self.corpus = corpus
        self.mu = mu
        self.collection_len = sum(len(doc) for doc in corpus)
        base_dir = os.getcwd()
        csv_path = os.path.join(base_dir, "..", "data", "archive-5", "archive (2)", "bbc-news-data.csv")
        self.df = pd.read_csv(csv_path, sep="\t")

    def _collection_prob(self, q_term):
        freq = 0
        for doc in self.corpus:
            freq += doc.count(q_term)
        return freq / self.collection_len

    def _score(self, doc, query):
        total_score = 0
        for q_term in query:
            term_freq = doc.count(q_term)
            coll_prob = self._collection_prob(q_term)
            smoothing_prob = (term_freq + self.mu * coll_prob) / (len(doc) + self.mu)

            if smoothing_prob > 0:
                total_score += math.log(smoothing_prob )

        return total_score

    def rank(self,query):
        scores = []

        for article in self.corpus:
            scores.append(self._score(doc=article,query=query))
        self.df['scores'] = scores
        return self.df.sort_values("scores",ascending=False)

In [9]:
uniLM = UnigramLM(corpus=processed_corpus, mu= 2000)

In [10]:
query = "rising fuel prices in europe"
query_prepro = BM25_PreProcess(corpus=[query],set_stemming=True,set_lemmatization=False,set_stopwords=True)
preprocessed_query = query_prepro.get_corpus()[0]

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/arturaliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
scores = uniLM.rank(preprocessed_query)
scores

,category,filename,title,content,scores
11,business,012.txt,Indonesians face fuel price rise,Indonesia's government has confirmed it is co...,-24.439950
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,-24.900265
44,business,045.txt,US crude prices surge above $53,US crude prices have soared to fresh four-mon...,-25.526380
269,business,270.txt,Oil prices reach three-month low,Oil prices have fallen heavily for a second d...,-25.644383
151,business,152.txt,Crude oil prices back above $50,Cold weather across parts of the United State...,-25.728433
...,...,...,...,...,...
1832,tech,009.txt,Apple laptop is 'greatest gadget',The Apple Powerbook 100 has been chosen as th...,-30.749676
1803,sport,491.txt,Nadal puts Spain 2-0 up,Result: Nadal 6-7 (6/8) 6-2 7-6 (8/6) 6-2 Rod...,-30.900582
2224,tech,401.txt,Losing yourself in online gaming,"Online role playing games are time-consuming,...",-31.316417
1185,politics,290.txt,Terror powers expose 'tyranny',The Lord Chancellor has defended government p...,-31.632127
